# 02c - DNS record and TLSA collection (resumable)

**Run all; if the runtime dies, run all again.** Same ledger pattern as the
certificate probe: local SQLite state, Drive backup every batch, shards
flushed as they fill.

For every domain in the probe universe this collects A/AAAA/NS/MX records,
the DNSSEC AD flag from a validating resolver, and the TLSA record for
`_443._tcp`, plus per-lookup wall-clock. Two things depend on it:

* the **DANE operational experiment** (notebook 14): with TLSA presence and
  DNSSEC status per domain, the four resolver policies - no ML, DANE only, ML
  only, ML + DANE - can be simulated on the frozen test set;
* the **DNS feature group** in `features.yaml`, which was designed but never
  populated.

Lookups go to validating public resolvers (Cloudflare 1.1.1.1 and Google
8.8.8.8, alternating) so the AD flag is meaningful. ~6 queries per domain at
64 parallel workers: roughly 1-2 hours for 50k domains.

In [ ]:
# --- standard header ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'
if os.path.isdir(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)
else:
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone','-q',f'https://{TOKEN}@{URL}',REPO], check=True)
sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
!pip -q install dnspython pyarrow zstandard

In [ ]:
import pandas as pd, numpy as np, time, concurrent.futures, dns.resolver
from pathlib import Path
from src.collect.ledger import Ledger
from src.collect import dns_records
from src.utils.io import ShardWriter, read_shards
from src.utils.logging_setup import get_logger

log = get_logger('collect_dns', P['artifacts']['logs'])
local_dir = Path(P['local']['ledger']).parent
ledger = Ledger(local_dir/'dns_ledger.db', drive_backup=f"{P['artifacts']['logs']}/dns_ledger.db")
print('restored from Drive backup:', ledger.restore_from_backup())

universe = pd.read_parquet(f"{P['data']['interim']}/probe_universe.parquet")
ledger.enqueue(universe[['domain','source','label']].assign(label=universe['label'].astype(str)).to_dict('records'))
print(ledger.summary())

In [ ]:
RESOLVERS = [['1.1.1.1','1.0.0.1'], ['8.8.8.8','8.8.4.4']]
def make_resolver(i):
    r = dns.resolver.Resolver(configure=False); r.nameservers = RESOLVERS[i % 2]
    r.timeout = 4.0; r.lifetime = 8.0
    r.use_edns(0, dns.flags.DO, 1232)      # request DNSSEC so AD is set by a validating resolver
    return r
import dns.flags

def one(args):
    i, domain = args
    t0 = time.perf_counter()
    try:
        row = dns_records.collect(domain, resolver=make_resolver(i))
        row['lookup_ms'] = (time.perf_counter()-t0)*1000
        status = 'success' if (row.get('n_a_records',0) or row.get('has_aaaa') or row.get('n_ns_records',0)) else 'nxdomain'
        return domain, status, row, None
    except Exception as e:
        return domain, 'dns_error', None, str(e)[:200]

OUT = f"{P['data']['collected']}/dns_records"
writer = ShardWriter(OUT, 'dns', rows_per_shard=2000)
batch = 0
try:
    while True:
        todo = ledger.pending(limit=1000)
        if not todo: break
        with concurrent.futures.ThreadPoolExecutor(64) as pool:
            for domain, status, row, err in pool.map(one, list(enumerate(todo))):
                if row: writer.add(row)
                ledger.mark(domain, status, shard_id=writer._shard_id, error=err)
        writer.flush(); ledger.commit(backup=True); batch += 1
        log.info('DNS batch %d | %s', batch, ledger.summary())
finally:
    writer.flush(); ledger.commit(backup=True)
ledger.retire_exhausted(); ledger.backup()
print(ledger.summary())

## Coverage - the numbers that motivate the trust score

In [ ]:
dnsr = read_shards(OUT, 'dns').drop_duplicates('domain', keep='last')
d = universe.merge(dnsr, on='domain', how='left')
d['resolves'] = d['n_a_records'].fillna(0) > 0
cov = d.groupby(['label','source']).agg(n=('domain','size'), resolves=('resolves','mean'),
        dnssec=('dnssec_signed', lambda s: s.fillna(False).mean()),
        tlsa=('has_tlsa', lambda s: s.fillna(False).mean()),
        lookup_ms_median=('lookup_ms','median')).round(4)
display(cov); cov.to_csv(Path(P['results']['tables'])/'table_dns_tlsa_coverage.csv')
print('domains with a TLSA record:', int(d['has_tlsa'].fillna(False).sum()),
      '| DNSSEC-signed:', int(d['dnssec_signed'].fillna(False).sum()))

---
**Next:** notebook 14 (DANE operational simulation) reads these shards together
with the trust-score predictions.